# LifeMate Dataset Analysis & Exploration (Zalo AI Challenge 2026)

Notebook này được thiết kế để chạy trên **Google Colab (hoặc Jupyter)**, kết nối trực tiếp với Google Drive tại đường dẫn:
`/content/drive/MyDrive/ZALO/train+public_test.zip`

### Mục tiêu chính:
1. **Mount Google Drive** an toàn và kiểm tra file nén `.zip`.
2. **Giải nén thông minh theo từng phần (Selective Extraction)**:
   - Bước A: Giải nén siêu dữ liệu (Metadata: `train.jsonl`, `public_test.jsonl`, các file catalog `*_tools_catalog.json`).
   - Bước B: Kiểm tra cấu trúc `*_environment/` và `zaic_tools/` (Tuân thủ luật thi không đọc `state.json`).
   - Bước C: Giải nén thư mục âm thanh `audio/` (gồm 5.036 WAV train + 1.933 WAV public_test).
3. **Phân tích Chuyên sâu Dữ liệu (EDA)**:
   - Thống kê phân bố độ dài âm thanh (hiện tại + history).
   - Phân tích độ phức tạp của chuỗi gọi tool (`tool_calls` gold trajectories).
   - Thống kê tỷ lệ các mẫu OOS (`stop`) và thiếu thông tin (`sys.ask_missing_info`).
   - Đối sánh độ lệch catalog giữa Train (46 apps, 147 tools) vs Public Test (36 apps, 122 tools — 11 apps mới).

## Phần 1: Kết nối Google Drive & Kiểm tra File Zip

In [1]:
import os
import zipfile
from pathlib import Path

# Kiểm tra môi trường Colab để mount drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = Path('/content/drive/MyDrive/ZALO/train+public_test.zip')
    WORK_DIR = Path('/content/dataset_lifemate')
except ImportError:
    # Fallback cho local hoặc server
    ZIP_PATH = Path('data/archives/train+public_test.zip')
    WORK_DIR = Path('data/raw')

print(f"Đường dẫn file zip: {ZIP_PATH}")
assert ZIP_PATH.exists(), f"KHÔNG tìm thấy file zip tại: {ZIP_PATH}! Hãy kiểm tra lại file trên Google Drive."

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"Kích thước file zip: {zip_size_mb:.2f} MB")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Thư mục làm việc giải nén: {WORK_DIR}")

Mounted at /content/drive
Đường dẫn file zip: /content/drive/MyDrive/ZALO/train+public_test.zip
Kích thước file zip: 835.83 MB
Thư mục làm việc giải nén: /content/dataset_lifemate


## Phần 2: Giải nén Từng Phần (Tránh đầy ổ đĩa & Tiết kiệm thời gian)

Thay vì giải nén toàn bộ 1.1GB ngay lập tức, ta giải nén trước **các file JSON / JSONL / README** để phân tích metadata chỉ trong vài giây.

In [2]:
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    all_files = zf.namelist()
    print(f"Tổng số file & thư mục bên trong zip: {len(all_files):,}")
    
    # 1. Lọc các file metadata nhẹ (JSONL, JSON, MD, VERSION, PY)
    meta_files = [
        f for f in all_files 
        if f.endswith(('.jsonl', '.json', '.md', 'VERSION')) or ('zaic_tools' in f and f.endswith('.py'))
    ]
    
    print(f"-> Đang giải nén {len(meta_files)} file metadata và thư viện zaic_tools...")
    zf.extractall(path=WORK_DIR, members=meta_files)
    print("Đã giải nén xong phần Metadata!")

Tổng số file & thư mục bên trong zip: 27,254
-> Đang giải nén 4572 file metadata và thư viện zaic_tools...
Đã giải nén xong phần Metadata!


In [3]:
# Tìm thư mục gốc sau khi giải nén (thường là train+public_test)
subdirs = [p for p in WORK_DIR.glob('*') if p.is_dir()]
DATASET_ROOT = WORK_DIR / 'train+public_test' if (WORK_DIR / 'train+public_test').exists() else WORK_DIR
print(f"Dataset Root: {DATASET_ROOT}")

for p in DATASET_ROOT.rglob('*'):
    if p.is_file():
        print(f"  - {p.relative_to(DATASET_ROOT)} ({p.stat().st_size:,} bytes)")

Dataset Root: /content/dataset_lifemate/train+public_test
  - VERSION (77 bytes)
  - README.md (9,848 bytes)
  - train/train_tools_catalog.json (91,423 bytes)
  - train/train.jsonl (7,198,841 bytes)
  - public_test/public_test_tools_catalog.json (76,805 bytes)
  - public_test/public_test.jsonl (1,593,830 bytes)
  - public_test/sample_submission.jsonl (607 bytes)
  - public_test/zaic_tools/executor.py (8,057 bytes)
  - public_test/zaic_tools/__init__.py (2,533 bytes)
  - public_test/zaic_tools/utils.py (2,872 bytes)
  - public_test/zaic_tools/__main__.py (2,623 bytes)
  - public_test/zaic_tools/validate.py (6,626 bytes)
  - public_test/zaic_tools/README.md (4,060 bytes)
  - public_test/public_test_environment/public_0228/state.json (8,627 bytes)
  - public_test/public_test_environment/public_0667/state.json (8,684 bytes)
  - public_test/public_test_environment/public_0620/state.json (11,339 bytes)
  - public_test/public_test_environment/public_0370/state.json (8,101 bytes)
  - public_te

## Phần 3: Phân tích Danh mục Công cụ (Catalog EDA)

So sánh sự dịch chuyển danh mục công cụ giữa **Train** và **Public Test** (Phát hiện 11 ứng dụng mới).

In [9]:
import json
import pandas as pd

train_catalog_path = list(DATASET_ROOT.rglob('*train_tools_catalog.json'))
public_catalog_path = list(DATASET_ROOT.rglob('*public_test_tools_catalog.json'))

assert public_catalog_path, "Không tìm thấy public_test_tools_catalog.json!"
with open(public_catalog_path[0], 'r', encoding='utf-8') as f:
    pub_catalog = json.load(f)

catalog_records = []
for app in pub_catalog.get('apps', []):
    app_id = app.get('id') or (app.get('tools', [{}])[0].get('name', '').split('.')[0] if app.get('tools') else 'unknown')
    app_name_vi = app.get('name_vi', '')
    tools = app.get('tools', [])
    read_tools = [t['name'] for t in tools if t.get('kind') == 'read']
    write_tools = [t['name'] for t in tools if t.get('kind') == 'write']
    meta_tools = [t['name'] for t in tools if t.get('kind') == 'meta']
    catalog_records.append({
        'app_id': app_id,
        'name_vi': app_name_vi,
        'group': app.get('group', ''),
        'total_tools': len(tools),
        'read_tools_count': len(read_tools),
        'write_tools_count': len(write_tools),
        'read_tools': ', '.join([t.split('.')[-1] for t in read_tools]),
        'write_tools': ', '.join([t.split('.')[-1] for t in write_tools])
    })

df_catalog = pd.DataFrame(catalog_records)
print(f"Tổng số App: {len(df_catalog)} | Tổng số Tool: {df_catalog['total_tools'].sum()}")
display(df_catalog.sort_values(by='total_tools', ascending=False).reset_index(drop=True))

Tổng số App: 36 | Tổng số Tool: 122


,app_id,name_vi,group,total_tools,read_tools_count,write_tools_count,read_tools,write_tools
0,smart_home,Nhà thông minh,home,6,1,5,get_status,"light_set, fan_set, climate_set, switch_set, s..."
1,email,Email,communication,6,2,4,"list, read","send, create_draft, reply, archive"
2,ecommerce,Mua sắm online,shopping,6,2,4,"search, order_status","add_to_cart, checkout, cancel_order, request_r..."
3,banking,Ngân hàng,finance,5,3,2,"get_balance, list_transactions, list_beneficia...","transfer, open_savings"
4,food_delivery,Đặt đồ ăn,food,5,3,2,"list_restaurants, get_menu, order_history","place_order, cancel_order"
5,flights,Vé máy bay,travel,5,2,3,"search, my_bookings","book, cancel, check_in"
6,chat_ott,Nhắn tin OTT,communication,5,2,3,"list_chats, read_chat","send_message, send_location, call"
7,alarm,Báo thức,planning,4,1,3,list,"create, update, delete"
8,calendar,Lịch,planning,4,1,3,list_events,"create_event, update_event, cancel_event"
9,reminders,Nhắc việc,planning,4,1,3,list,"create, complete, delete"


## Phần 4: Phân tích Tập Huấn luyện (`train.jsonl`)

Khảo sát cấu trúc của 1.545 mẫu trong tập `train.jsonl`:
- Độ dài câu thoại (`audio_text`).
- Số lượng ứng dụng được cài đặt (`installed_apps`).
- Số lượt hội thoại trong lịch sử (`history`).
- Số bước gọi tool của đáp án chuẩn (`label.tool_calls`).

In [5]:
train_jsonl_paths = list(DATASET_ROOT.rglob('*train.jsonl'))
if train_jsonl_paths:
    train_file = train_jsonl_paths[0]
    samples = []
    with open(train_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                samples.append(json.loads(line))
    print(f"Tổng số mẫu trong train.jsonl: {len(samples):,}")
    
    train_stats = []
    for s in samples:
        installed_apps = s.get('context', {}).get('installed_apps', {})
        history = s.get('history', [])
        label_calls = s.get('label', {}).get('tool_calls', [])
        
        has_ask_missing = any(c.get('name') == 'sys.ask_missing_info' for c in label_calls)
        
        train_stats.append({
            'id': s.get('id'),
            'query_length_chars': len(s.get('audio_text', '')),
            'query_word_count': len(s.get('audio_text', '').split()),
            'installed_apps_count': len(installed_apps),
            'installed_tools_count': sum(len(a.get('tools', [])) for a in installed_apps.values()),
            'history_turns': len(history),
            'gold_tool_calls_count': len(label_calls),
            'is_oos_stop': len(label_calls) == 0,
            'is_ask_missing_info': has_ask_missing
        })
    
    df_train = pd.DataFrame(train_stats)
    print("--- THỐNG KÊ MÔ TẢ TẬP TRAIN ---")
    display(df_train.describe().round(2))
else:
    print("Chưa giải nén train.jsonl, hãy kiểm tra lại file zip!")

Tổng số mẫu trong train.jsonl: 1,545
--- THỐNG KÊ MÔ TẢ TẬP TRAIN ---


,query_length_chars,query_word_count,installed_apps_count,installed_tools_count,history_turns,gold_tool_calls_count
count,1545.00,1545.00,1545.00,1545.00,1545.00,1545.00
mean,97.32,22.45,9.20,32.53,4.52,2.29
std,35.04,7.91,2.82,10.06,5.77,1.35
min,12.00,3.00,3.00,8.00,0.00,0.00
25%,76.00,18.00,7.00,25.00,0.00,1.00
50%,96.00,22.00,9.00,31.00,2.00,2.00
75%,115.00,26.00,11.00,39.00,6.00,3.00
max,296.00,67.00,18.00,62.00,24.00,6.00


In [6]:
if 'df_train' in locals():
    print("=== PHÂN BỐ SỐ BƯỚC GỌI TOOL (GOLD TRAJECTORIES) ===")
    display(df_train['gold_tool_calls_count'].value_counts().sort_index().to_frame('Số mẫu'))
    
    print("\n=== TỶ LỆ CÁC TÌNH HUỐNG ĐẶC BIỆT ===")
    print(f"- Số mẫu OOS / Dừng ngay (0 call): {df_train['is_oos_stop'].sum()} ({df_train['is_oos_stop'].mean()*100:.1f}%)")
    print(f"- Số mẫu thiếu thông tin (sys.ask_missing_info): {df_train['is_ask_missing_info'].sum()} ({df_train['is_ask_missing_info'].mean()*100:.1f}%)")
    print(f"- Số mẫu có lịch sử hội thoại (History > 0): {(df_train['history_turns'] > 0).sum()} ({(df_train['history_turns'] > 0).mean()*100:.1f}%)")

=== PHÂN BỐ SỐ BƯỚC GỌI TOOL (GOLD TRAJECTORIES) ===


,Số mẫu
gold_tool_calls_count,
0,111
1,369
2,428
3,359
4,161
5,107
6,10



=== TỶ LỆ CÁC TÌNH HUỐNG ĐẶC BIỆT ===
- Số mẫu OOS / Dừng ngay (0 call): 111 (7.2%)
- Số mẫu thiếu thông tin (sys.ask_missing_info): 217 (14.0%)
- Số mẫu có lịch sử hội thoại (History > 0): 1018 (65.9%)


### In chi tiết 1 mẫu cụ thể trong tập Train (Ví dụ minh họa đầy đủ các trường)

In [10]:
# Tìm một mẫu tiêu biểu có cả history và nhiều hơn 1 bước gọi tool trong train
if 'samples' in locals() and samples:
    selected = None
    for s in samples:
        if len(s.get('history', [])) > 0 and len(s.get('label', {}).get('tool_calls', [])) >= 2:
            selected = s
            break
    if not selected:
        selected = samples[0]
    
    print(f"=== CHI TIẾT SAMPLE ID: {selected.get('id')} ===\n")
    print(f"1. Yêu cầu âm thanh (audio): {selected.get('audio')}")
    print(f"2. Lời thoại người dùng (audio_text):\n   -> '{selected.get('audio_text')}'\n")
    
    print("3. Ngữ cảnh (context):")
    print(f"   - Thời gian hiện tại: {selected.get('context', {}).get('current_time')}")
    print(f"   - Múi giờ: {selected.get('context', {}).get('timezone')}")
    print(f"   - Vị trí: {selected.get('context', {}).get('location')}")
    print(f"   - Các ứng dụng cài đặt (installed_apps): {list(selected.get('context', {}).get('installed_apps', {}).keys())}\n")
    
    print(f"4. Lịch sử hội thoại (history: {len(selected.get('history', []))} lượt):")
    for i, turn in enumerate(selected.get('history', [])):
        role = turn.get('role')
        if role == 'user':
            print(f"   [Turn {i+1} - USER]: {turn.get('audio_text', turn.get('audio'))}")
        else:
            print(f"   [Turn {i+1} - BOT]: {turn.get('text')}")
            if turn.get('tool_calls'):
                print(f"       -> Tool đã gọi: {json.dumps(turn.get('tool_calls'), ensure_ascii=False)}")
            if turn.get('tool_results'):
                print(f"       -> Kết quả trả về: {json.dumps(turn.get('tool_results'), ensure_ascii=False)}")
    
    print("\n5. Đáp án chuẩn của BTC (label):")
    print("   * Gold tool_calls (chuỗi lệnh chuẩn):")
    for step_idx, call in enumerate(selected.get('label', {}).get('tool_calls', [])):
        res = selected.get('label', {}).get('tool_results', [])[step_idx] if step_idx < len(selected.get('label', {}).get('tool_results', [])) else {}
        print(f"     Bước {step_idx + 1}: Gọi '{call.get('name')}' với tham số {json.dumps(call.get('arguments'), ensure_ascii=False)}")
        print(f"             -> Kết quả nhận được: {json.dumps(res, ensure_ascii=False)}")
    
    print("\n6. Dạng JSON thô đầy đủ của sample này:")
    print(json.dumps(selected, ensure_ascii=False, indent=2))
else:
    print("Chưa có biến samples, hãy chạy cell đọc train.jsonl phía trên trước!")

=== CHI TIẾT SAMPLE ID: train_0069 ===

1. Yêu cầu âm thanh (audio): audio/train_0069/train_0069.wav
2. Lời thoại người dùng (audio_text):
   -> 'Tắt đèn phòng ngủ nhỏ giúp chị, rồi gửi tin nhắn cho Hương là nhớ mang hộ cái sạc nhé.'

3. Ngữ cảnh (context):
   - Thời gian hiện tại: 2026-11-20T21:40:00+07:00
   - Múi giờ: Asia/Ho_Chi_Minh
   - Vị trí: {'lat': 16.0544, 'lng': 108.2022}
   - Các ứng dụng cài đặt (installed_apps): ['smart_home', 'contacts', 'sms', 'gold_price', 'ride_hailing', 'banking', 'music', 'medication', 'tv', 'calendar', 'alarm', 'phone', 'ewallet', 'food_delivery']

4. Lịch sử hội thoại (history: 10 lượt):
   [Turn 1 - USER]: Gần đây có quán bún bò nào ngon không?
   [Turn 2 - BOT]: Có Bún bò Huế Cô Ba.
       -> Tool đã gọi: [{"name": "food_delivery.list_restaurants", "arguments": {}}]
       -> Kết quả trả về: [{"ok": true, "restaurants": [{"id": "rest_01", "name": "Bún bò Huế Cô Ba", "tags": ["bún bò", "món Huế"], "rating": 4.5}, {"id": "rest_02", "name": "Cơm t

### Thống kê Các dạng Câu hỏi & Ý định (Intents / Linguistic Patterns) trong `audio_text`

Phân loại 1.545 câu nói của người dùng theo:
1. **Hành động ngữ nghĩa chính**: Tra cứu/Kiểm tra (`read`), Thao tác/Đặt/Tạo (`create`), Cập nhật/Dời lịch (`update`), Xóa/Hủy (`delete`), Chào hỏi/Xã giao (`oos`).
2. **Hiện tượng Ngôn ngữ Đa lượt**: Tỉnh lược/Đại từ chỉ định (*'đơn đó'*, *'vừa nãy'*, *'người đó'* - DUR), Đổi ý định (*'đổi lại'*, *'dời sang'*, *'hủy giúp'*, *'thôi không'* - SUPERSEDES).
3. **Mức độ phức tạp**: Câu đơn (1 hành động) vs Câu ghép (nhiều hành động kết nối bởi *'rồi'*, *'sau đó'*, *'đồng thời'*).

In [11]:
import re
import pandas as pd

if 'samples' in locals() and samples:
    pattern_stats = {
        'tra_cuu_kiem_tra': 0,
        'tao_moi_thiet_lap': 0,
        'cap_nhat_chinh_sua': 0,
        'xoa_huy_bo': 0,
        'cau_ghep_da_hanh_dong': 0,
        'chua_dai_tu_tinh_luoc': 0,
        'chua_doi_y_dinh': 0,
        'chao_hoi_xa_giao': 0
    }
    
    re_anaphora = re.compile(r'\b(vừa nãy|hồi nãy|lúc nãy|ở trên|đó|đấy|này|cái đó|người đó|anh ấy|chị ấy)\b', re.I)
    re_supersedes = re.compile(r'\b(đổi lại|dời sang|sửa lại|thay đổi|hủy giúp|thôi không|khoan đã|à nhầm)\b', re.I)
    re_compound = re.compile(r'\b(rồi|sau đó|đồng thời|tiện thể|kèm theo|tiếp theo)\b', re.I)
    re_lookup = re.compile(r'\b(xem|kiểm tra|tìm|tra|cho tôi biết|bao nhiêu|mấy giờ|có không|danh sách)\b', re.I)
    re_create = re.compile(r'\b(đặt|tạo|thêm|bật|mở|gửi|chuyển|mua|book|gọi cho)\b', re.I)
    re_update = re.compile(r'\b(sửa|đổi|dời|chuyển sang|cập nhật|tăng|giảm|chỉnh)\b', re.I)
    re_delete = re.compile(r'\b(xóa|hủy|tắt|đóng|ngưng|bỏ)\b', re.I)
    re_chit_chat = re.compile(r'\b(chào|cảm ơn|thank|tạm biệt|bạn tên gì|giỏi lắm)\b', re.I)
    
    for s in samples:
        text = s.get('audio_text', '').lower()
        if re_lookup.search(text): pattern_stats['tra_cuu_kiem_tra'] += 1
        if re_create.search(text): pattern_stats['tao_moi_thiet_lap'] += 1
        if re_update.search(text): pattern_stats['cap_nhat_chinh_sua'] += 1
        if re_delete.search(text): pattern_stats['xoa_huy_bo'] += 1
        if re_compound.search(text): pattern_stats['cau_ghep_da_hanh_dong'] += 1
        if re_anaphora.search(text): pattern_stats['chua_dai_tu_tinh_luoc'] += 1
        if re_supersedes.search(text): pattern_stats['chua_doi_y_dinh'] += 1
        if re_chit_chat.search(text) and len(s.get('label', {}).get('tool_calls', [])) == 0:
            pattern_stats['chao_hoi_xa_giao'] += 1
            
    total_s = len(samples)
    df_patterns = pd.DataFrame([
        {'Dạng câu hỏi / Ý định ngôn ngữ': '1. Tra cứu / Xem thông tin (Lookup / Read)', 'Số mẫu': pattern_stats['tra_cuu_kiem_tra'], 'Tỷ lệ (%)': round(pattern_stats['tra_cuu_kiem_tra']/total_s*100, 1)},
        {'Dạng câu hỏi / Ý định ngôn ngữ': '2. Tạo mới / Kích hoạt hành động (Create / Write)', 'Số mẫu': pattern_stats['tao_moi_thiet_lap'], 'Tỷ lệ (%)': round(pattern_stats['tao_moi_thiet_lap']/total_s*100, 1)},
        {'Dạng câu hỏi / Ý định ngôn ngữ': '3. Cập nhật / Dời lịch / Chỉnh sửa (Update / Write)', 'Số mẫu': pattern_stats['cap_nhat_chinh_sua'], 'Tỷ lệ (%)': round(pattern_stats['cap_nhat_chinh_sua']/total_s*100, 1)},
        {'Dạng câu hỏi / Ý định ngôn ngữ': '4. Xóa / Hủy bỏ / Tắt thiết bị (Delete / Cancel)', 'Số mẫu': pattern_stats['xoa_huy_bo'], 'Tỷ lệ (%)': round(pattern_stats['xoa_huy_bo']/total_s*100, 1)},
        {'Dạng câu hỏi / Ý định ngôn ngữ': '5. Câu ghép đa hành động (Có từ nối: rồi, sau đó...)', 'Số mẫu': pattern_stats['cau_ghep_da_hanh_dong'], 'Tỷ lệ (%)': round(pattern_stats['cau_ghep_da_hanh_dong']/total_s*100, 1)},
        {'Dạng câu hỏi / Ý định ngôn ngữ': '6. Câu chứa đại từ chỉ định / Tỉnh lược (DUR Context)', 'Số mẫu': pattern_stats['chua_dai_tu_tinh_luoc'], 'Tỷ lệ (%)': round(pattern_stats['chua_dai_tu_tinh_luoc']/total_s*100, 1)},
        {'Dạng câu hỏi / Ý định ngôn ngữ': '7. Người dùng đổi ý định giữa chừng (SUPERSEDES)', 'Số mẫu': pattern_stats['chua_doi_y_dinh'], 'Tỷ lệ (%)': round(pattern_stats['chua_doi_y_dinh']/total_s*100, 1)},
        {'Dạng câu hỏi / Ý định ngôn ngữ': '8. Chào hỏi / Xã giao / Không gọi tool (OOS - Stop)', 'Số mẫu': pattern_stats['chao_hoi_xa_giao'], 'Tỷ lệ (%)': round(pattern_stats['chao_hoi_xa_giao']/total_s*100, 1)}
    ])
    
    print(f"=== BẢNG THỐNG KÊ CÁC DẠNG CÂU HỎI TRONG AUDIO_TEXT (TẬP TRAIN: {total_s:,} MẪU) ===")
    display(df_patterns)
else:
    print("Chưa có biến samples, hãy chạy cell đọc train.jsonl phía trên!")

=== BẢNG THỐNG KÊ CÁC DẠNG CÂU HỎI TRONG AUDIO_TEXT (TẬP TRAIN: 1,545 MẪU) ===


,Dạng câu hỏi / Ý định ngôn ngữ,Số mẫu,Tỷ lệ (%)
0,1. Tra cứu / Xem thông tin (Lookup / Read),219,14.2
1,2. Tạo mới / Kích hoạt hành động (Create / Write),746,48.3
2,3. Cập nhật / Dời lịch / Chỉnh sửa (Update / W...,153,9.9
3,4. Xóa / Hủy bỏ / Tắt thiết bị (Delete / Cancel),210,13.6
4,"5. Câu ghép đa hành động (Có từ nối: rồi, sau ...",494,32.0
5,6. Câu chứa đại từ chỉ định / Tỉnh lược (DUR C...,524,33.9
6,7. Người dùng đổi ý định giữa chừng (SUPERSEDES),17,1.1
7,8. Chào hỏi / Xã giao / Không gọi tool (OOS - ...,6,0.4


### Trực quan hóa Mẫu Tiêu biểu (2 Samples cho mỗi Dạng trong 8 Dạng)

In [12]:
if 'samples' in locals() and samples:
    pattern_definitions = [
        ('1. Tra cứu / Xem thông tin (Lookup / Read)', re.compile(r'\b(xem|kiểm tra|tìm|tra|cho tôi biết|bao nhiêu|mấy giờ|có không|danh sách)\b', re.I), lambda s: len(s.get('label', {}).get('tool_calls', [])) > 0),
        ('2. Tạo mới / Kích hoạt hành động (Create / Write)', re.compile(r'\b(đặt|tạo|thêm|bật|mở|gửi|chuyển|mua|book|gọi cho)\b', re.I), lambda s: any(c.get('name', '').endswith(('.create', '.create_event', '.send', '.transfer', '.light_set', '.fan_set', '.book')) for c in s.get('label', {}).get('tool_calls', []))),
        ('3. Cập nhật / Dời lịch / Chỉnh sửa (Update / Write)', re.compile(r'\b(sửa|đổi|dời|chuyển sang|cập nhật|tăng|giảm|chỉnh)\b', re.I), lambda s: any(c.get('name', '').endswith(('.update', '.update_event', '.tune', '.append')) for c in s.get('label', {}).get('tool_calls', []))),
        ('4. Xóa / Hủy bỏ / Tắt thiết bị (Delete / Cancel)', re.compile(r'\b(xóa|hủy|tắt|đóng|ngưng|bỏ)\b', re.I), lambda s: any(c.get('name', '').endswith(('.delete', '.cancel', '.cancel_event', '.cancel_order')) or 'power": "off"' in json.dumps(c) for c in s.get('label', {}).get('tool_calls', []))),
        ('5. Câu ghép đa hành động (Có từ nối: rồi, sau đó...)', re.compile(r'\b(rồi|sau đó|đồng thời|tiện thể|kèm theo|tiếp theo)\b', re.I), lambda s: len(s.get('label', {}).get('tool_calls', [])) >= 2),
        ('6. Câu chứa đại từ chỉ định / Tỉnh lược (DUR Context)', re.compile(r'\b(vừa nãy|hồi nãy|lúc nãy|ở trên|đó|đấy|này|cái đó|người đó|anh ấy|chị ấy)\b', re.I), lambda s: len(s.get('history', [])) > 0),
        ('7. Người dùng đổi ý định giữa chừng (SUPERSEDES)', re.compile(r'\b(đổi lại|dời sang|sửa lại|thay đổi|hủy giúp|thôi không|khoan đã|à nhầm)\b', re.I), lambda s: True),
        ('8. Chào hỏi / Xã giao / Không gọi tool (OOS - Stop)', re.compile(r'\b(chào|cảm ơn|thank|tạm biệt|bạn tên gì|giỏi lắm)\b', re.I), lambda s: len(s.get('label', {}).get('tool_calls', [])) == 0)
    ]
    
    for title, regex, cond in pattern_definitions:
        matched = []
        for s in samples:
            if regex.search(s.get('audio_text', '')) and cond(s):
                matched.append(s)
                if len(matched) == 2:
                    break
        
        print(f"========================================================================================")
        print(f"📌 DẠNG: {title}")
        print(f"========================================================================================")
        for idx, s in enumerate(matched):
            calls_str = ' -> '.join([f"{c['name']}({list(c.get('arguments', {}).keys())})" for c in s.get('label', {}).get('tool_calls', [])]) or "{'action': 'stop'} (Không gọi tool)"
            print(f"[Ví dụ {idx + 1}] ID: {s.get('id')}")
            print(f"  • Lời thoại (audio_text): \"{s.get('audio_text')}\"")
            print(f"  • Số lượt history: {len(s.get('history', []))}")
            print(f"  • Chuỗi tool chuẩn: {calls_str}")
            print("-" * 80)
        print()
else:
    print("Chưa có biến samples!")

📌 DẠNG: 1. Tra cứu / Xem thông tin (Lookup / Read)
[Ví dụ 1] ID: train_0002
  • Lời thoại (audio_text): "Nếu mai mưa thì hủy lịch đá bóng... À mà thôi, cứ để lịch đó, chỉ cần xem giúp mai có mưa không."
  • Số lượt history: 0
  • Chuỗi tool chuẩn: weather.get_forecast(['date'])
--------------------------------------------------------------------------------
[Ví dụ 2] ID: train_0050
  • Lời thoại (audio_text): "Alô, cho em Linh hỏi giá vàng SJC hôm nay bao nhiêu dùm nha."
  • Số lượt history: 0
  • Chuỗi tool chuẩn: gold_price.get_price(['type'])
--------------------------------------------------------------------------------

📌 DẠNG: 2. Tạo mới / Kích hoạt hành động (Create / Write)
[Ví dụ 1] ID: train_0003
  • Lời thoại (audio_text): "Ơi, nạp giúp mình 50 nghìn điện thoại cho bác Ba với chuyển ví cho cô Mai số tiền điện tháng này nhé."
  • Số lượt history: 0
  • Chuỗi tool chuẩn: bill_payment.list_bills([]) -> contacts.search(['query']) -> ewallet.topup_mobile(['phone', 'amount']) -> 

## Phần 5: Phân tích Tập Kiểm thử Công khai (`public_test.jsonl`)

Khảo sát các trường hợp của tập `public_test` (Lưu ý: Không có trường `audio_text` và `label`).

In [7]:
pub_jsonl_paths = list(DATASET_ROOT.rglob('*public_test.jsonl'))
if pub_jsonl_paths:
    pub_file = pub_jsonl_paths[0]
    pub_samples = []
    with open(pub_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                pub_samples.append(json.loads(line))
    print(f"Tổng số mẫu trong public_test.jsonl: {len(pub_samples):,}")
    
    pub_stats = []
    for s in pub_samples:
        installed_apps = s.get('context', {}).get('installed_apps', {})
        history = s.get('history', [])
        pub_stats.append({
            'id': s.get('id'),
            'installed_apps_count': len(installed_apps),
            'installed_tools_count': sum(len(a.get('tools', [])) for a in installed_apps.values()),
            'history_turns': len(history)
        })
    df_pub = pd.DataFrame(pub_stats)
    print("--- THỐNG KÊ MÔ TẢ PUBLIC TEST ---")
    display(df_pub.describe().round(2))
else:
    print("Chưa tìm thấy public_test.jsonl!")

Tổng số mẫu trong public_test.jsonl: 634
--- THỐNG KÊ MÔ TẢ PUBLIC TEST ---


,installed_apps_count,installed_tools_count,history_turns
count,634.00,634.00,634.00
mean,8.53,30.66,4.10
std,2.69,9.90,5.34
min,3.00,10.00,0.00
25%,7.00,23.00,0.00
50%,8.00,30.00,2.00
75%,10.00,37.00,6.00
max,17.00,59.00,24.00


## Phần 6: Giải nén Thư mục Âm thanh (`audio/`) Tùy chọn (Cho Huấn luyện ASR / Full Pipeline)

Thư mục `audio/` chứa hàng ngàn file `.wav` 16kHz mono. Nếu bạn chỉ muốn phân tích text thì có thể bỏ qua bước này. Nếu cần chạy kiểm thử giọng nói với PhoWhisper, hãy chạy cell dưới đây.

In [8]:
# Chạy cell này nếu bạn muốn giải nén toàn bộ các file âm thanh WAV
EXTRACT_AUDIO = False  # Đổi thành True nếu bạn muốn giải nén WAV

if EXTRACT_AUDIO:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        audio_files = [f for f in zf.namelist() if f.endswith('.wav')]
        print(f"Tìm thấy {len(audio_files):,} file WAV trong zip.")
        print("Đang tiến hành giải nén audio (quá trình này mất khoảng 1-2 phút)...")
        zf.extractall(path=WORK_DIR, members=audio_files)
        print("Đã giải nén xong toàn bộ audio WAV!")
else:
    print("EXTRACT_AUDIO đang tắt. Đặt EXTRACT_AUDIO = True nếu cần giải nén file âm thanh.")

EXTRACT_AUDIO đang tắt. Đặt EXTRACT_AUDIO = True nếu cần giải nén file âm thanh.
